In [1]:
import os
os.chdir("/home/jupyter/fire-impact-curves/notebooks")
import sys
sys.path.append("/home/jupyter/fire-impact-curves")

import pandas as pd
import numpy as np
import rasterio
from rasterio.features import rasterize
from src.grids import read_window, reproject_to_grid, pad_bounds
from src.fires import load_fire, inner_burn_area, utm_epsg_for
from src.pipeline import DNBR_DROP_BELOW
from src.curves import fit_group, logistic

full = pd.read_parquet("../data/processed/all_cells_current.parquet")

# reconstruct paths for the 8 fires already downloaded, no EE needed, files are on disk
done_fires = ['2020_112', '2020_135', '2020_464', '2021_190', '2020_838262', '2021_13058', '2021_12957', '2021_12806']
before_year = {uid: int(uid[:4]) - 1 for uid in done_fires}
after_years = {
    "2020_112": [2021, 2022], "2020_135": [2021, 2022], "2020_464": [2021, 2022],
    "2020_838262": [2021, 2022], "2021_190": [2022], "2021_13058": [2022],
    "2021_12957": [2022], "2021_12806": [2022],
}
stability_paths = {}
for uid in done_fires:
    stability_paths[uid] = {yr: f"../data/agb_stability/agb_{uid}_{yr}.tif" for yr in [2015, 2017, 2018]}
    stability_paths[uid][before_year[uid]] = f"../data/cci_agb_ee/agb_{uid}_{before_year[uid]}.tif"

print("reconstructed paths for", len(stability_paths), "fires")

reconstructed paths for 8 fires


In [2]:
def stability_cv_for_fire(uid, year_paths, agb_after_years, min_agb_before=10):
    fire = load_fire(uid)
    bounds = pad_bounds(fire.total_bounds, 0.15)
    lon, lat = fire.geometry.iloc[0].centroid.x, fire.geometry.iloc[0].centroid.y
    utm_epsg = utm_epsg_for(lon, lat)
    inner = inner_burn_area(fire, utm_epsg=utm_epsg)

    years = sorted(year_paths.keys())
    before_yr = years[-1]
    arrays, tr_ref, crs_ref, shape = {}, None, None, None
    for yr in years:
        arr, tr, crs = read_window(year_paths[yr], bounds)
        arrays[yr] = arr
        if tr_ref is None:
            tr_ref, crs_ref, shape = tr, crs, arr.shape

    mask_inner = rasterize(
        [(g, 1) for g in inner.geometry], out_shape=shape,
        transform=tr_ref, fill=0, dtype="uint8",
    ).astype(bool)
    valid = np.isfinite(arrays[before_yr]) & (arrays[before_yr] > min_agb_before)
    for yr in agb_after_years:
        after_arr, _, _ = read_window(f"../data/cci_agb_ee/agb_{uid}_{yr}.tif", bounds)
        valid &= np.isfinite(after_arr)
    keep = mask_inner & valid

    stack = np.stack([arrays[yr] for yr in years], axis=0)
    mean_agb = stack.mean(axis=0)
    std_agb = stack.std(axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        cv = std_agb / mean_agb

    severity_path = f"../data/severity/severity_{uid}.tif"
    dnbr_arr, tr_d, crs_d = read_window(severity_path, bounds, band=1)
    dnbr_100m = reproject_to_grid(
        np.nan_to_num(dnbr_arr, nan=-9999).astype("float32"), tr_d, crs_d, shape, tr_ref, crs_ref,
        src_nodata=-9999,
    )

    d = pd.DataFrame({"uid": uid, "cv": cv[keep], "dnbr_recomputed": dnbr_100m[keep]})
    d = d[d["dnbr_recomputed"] >= DNBR_DROP_BELOW].copy()
    return d

cv_results = [stability_cv_for_fire(uid, stability_paths[uid], after_years[uid]) for uid in done_fires]
cv_full = pd.concat(cv_results, ignore_index=True)

merged_parts = []
for uid in done_fires:
    sub_full = full[full["uid"] == uid].reset_index(drop=True)
    sub_cv = cv_full[cv_full["uid"] == uid].reset_index(drop=True)
    assert len(sub_full) == len(sub_cv), f"{uid}: mismatch {len(sub_full)} vs {len(sub_cv)}"
    merged = sub_full.copy()
    merged["cv"] = sub_cv["cv"].values
    merged_parts.append(merged)
merged_full = pd.concat(merged_parts, ignore_index=True)

grp = full[(full["height_class"] == "10-20m") & (full["forest_type_label"] == "evergreen_needleleaf")]
grp_with_cv = merged_full[(merged_full["height_class"] == "10-20m") & (merged_full["forest_type_label"] == "evergreen_needleleaf")]

rows = []
for uid in done_fires:
    train = grp[grp["uid"] != uid]
    fit = fit_group(train, min_fires=3, min_cells=200)
    if fit["status"] not in ("fitted", "degenerate_fit"):
        continue
    held = grp_with_cv[grp_with_cv["uid"] == uid]
    for sev, sev_grp in held.groupby("severity", observed=True):
        if len(sev_grp) == 0:
            continue
        x = sev_grp["dnbr"].mean()
        predicted = logistic(x, fit["l_max"], fit["k"], fit["x0"])
        actual_unweighted = sev_grp["loss2022"].mean()
        weights = 1 / (1 + sev_grp["cv"].clip(lower=0))
        actual_weighted = np.average(sev_grp["loss2022"], weights=weights)
        rows.append({
            "uid": uid, "severity": sev, "n_cells": len(sev_grp),
            "predicted": predicted,
            "residual_unweighted": actual_unweighted - predicted,
            "residual_weighted": actual_weighted - predicted,
        })

result = pd.DataFrame(rows)
print(result.to_string(index=False))
print()
print(f"MAE unweighted: {result['residual_unweighted'].abs().mean():.3f}")
print(f"MAE weighted:   {result['residual_weighted'].abs().mean():.3f}")

/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return

        uid severity  n_cells  predicted  residual_unweighted  residual_weighted
   2020_112     mild     4883   0.001050            -0.559979          -0.547309
   2020_112 moderate    16435   0.059335            -0.080568          -0.074124
   2020_112   severe     4953   0.160249             0.079930           0.086272
   2020_135     mild     2134   0.000161            -0.339686          -0.318635
   2020_135 moderate    14560   0.084689            -0.029512          -0.011607
   2020_135   severe    12768   0.155695             0.031857           0.042737
   2020_464 moderate       17   0.147616             0.382263           0.379098
   2020_464   severe       15   0.166384             0.500466           0.501707
   2021_190     mild     4679   0.000133            -0.056607          -0.037469
   2021_190 moderate    11268   0.024234             0.108752           0.129743
   2021_190   severe     2001   0.169281             0.055548           0.067903
2020_838262     mild       7

In [3]:
more_fires = ['2021_26579', '2021_26587', '2021_26580', '2021_26577', '2021_17228',
              '2021_18246', '2021_18206', '2020_6787', '2020_6783', '2021_53',
              '2020_6785', '2020_6781']

In [4]:
all_20_fires = done_fires + more_fires
before_year_20 = {uid: int(uid[:4]) - 1 for uid in all_20_fires}
after_years_20 = {uid: ([2021, 2022] if uid.startswith("2020") else [2022]) for uid in all_20_fires}

stability_paths_20 = {}
for uid in all_20_fires:
    stability_paths_20[uid] = {yr: f"../data/agb_stability/agb_{uid}_{yr}.tif" for yr in [2015, 2017, 2018]}
    stability_paths_20[uid][before_year_20[uid]] = f"../data/cci_agb_ee/agb_{uid}_{before_year_20[uid]}.tif"

cv_results_20 = [stability_cv_for_fire(uid, stability_paths_20[uid], after_years_20[uid]) for uid in all_20_fires]
cv_full_20 = pd.concat(cv_results_20, ignore_index=True)

merged_parts_20 = []
for uid in all_20_fires:
    sub_full = full[full["uid"] == uid].reset_index(drop=True)
    sub_cv = cv_full_20[cv_full_20["uid"] == uid].reset_index(drop=True)
    assert len(sub_full) == len(sub_cv), f"{uid}: mismatch {len(sub_full)} vs {len(sub_cv)}"
    merged = sub_full.copy()
    merged["cv"] = sub_cv["cv"].values
    merged_parts_20.append(merged)
merged_full_20 = pd.concat(merged_parts_20, ignore_index=True)

grp_with_cv_20 = merged_full_20[(merged_full_20["height_class"] == "10-20m") & (merged_full_20["forest_type_label"] == "evergreen_needleleaf")]

rows = []
for uid in all_20_fires:
    train = grp[grp["uid"] != uid]
    fit = fit_group(train, min_fires=3, min_cells=200)
    if fit["status"] not in ("fitted", "degenerate_fit"):
        continue
    held = grp_with_cv_20[grp_with_cv_20["uid"] == uid]
    for sev, sev_grp in held.groupby("severity", observed=True):
        if len(sev_grp) == 0:
            continue
        x = sev_grp["dnbr"].mean()
        predicted = logistic(x, fit["l_max"], fit["k"], fit["x0"])
        actual_unweighted = sev_grp["loss2022"].mean()
        weights = 1 / (1 + sev_grp["cv"].clip(lower=0))
        actual_weighted = np.average(sev_grp["loss2022"], weights=weights)
        rows.append({
            "uid": uid, "severity": sev, "n_cells": len(sev_grp),
            "residual_unweighted": actual_unweighted - predicted,
            "residual_weighted": actual_weighted - predicted,
        })

result_20 = pd.DataFrame(rows)
print(f"n={len(result_20)} fire-severity folds across {len(all_20_fires)} fires")
print(f"MAE unweighted: {result_20['residual_unweighted'].abs().mean():.3f}")
print(f"MAE weighted:   {result_20['residual_weighted'].abs().mean():.3f}")

/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return

n=52 fire-severity folds across 20 fires
MAE unweighted: 0.133
MAE weighted:   0.132


In [5]:
threshold = 0.3
rows = []
for uid in all_20_fires:
    train = grp[grp["uid"] != uid]
    fit = fit_group(train, min_fires=3, min_cells=200)
    if fit["status"] not in ("fitted", "degenerate_fit"):
        continue
    held = grp_with_cv_20[grp_with_cv_20["uid"] == uid]
    for sev, sev_grp in held.groupby("severity", observed=True):
        if len(sev_grp) == 0:
            continue
        x = sev_grp["dnbr"].mean()
        predicted = logistic(x, fit["l_max"], fit["k"], fit["x0"])

        actual_unweighted = sev_grp["loss2022"].mean()
        weights = 1 / (1 + sev_grp["cv"].clip(lower=0))
        actual_weighted = np.average(sev_grp["loss2022"], weights=weights)
        filtered = sev_grp[sev_grp["cv"] <= threshold]
        actual_filtered = filtered["loss2022"].mean() if len(filtered) > 0 else np.nan

        rows.append({
            "uid": uid, "severity": sev, "n_cells": len(sev_grp), "n_kept": len(filtered),
            "residual_unweighted": actual_unweighted - predicted,
            "residual_weighted": actual_weighted - predicted,
            "residual_filtered": (actual_filtered - predicted) if pd.notna(actual_filtered) else np.nan,
        })

result_20 = pd.DataFrame(rows)
print(f"n={len(result_20)} fire-severity folds across {len(all_20_fires)} fires")
print(f"MAE unweighted: {result_20['residual_unweighted'].abs().mean():.3f}")
print(f"MAE weighted:   {result_20['residual_weighted'].abs().mean():.3f}")
print(f"MAE filtered:   {result_20['residual_filtered'].abs().mean():.3f}")

n=52 fire-severity folds across 20 fires
MAE unweighted: 0.133
MAE weighted:   0.132
MAE filtered:   0.144
